In [1]:
from azureml.core import Workspace, Dataset, Datastore
from azureml.data.dataset_factory import DataPath
import pandas as pd

# اتصال به workspace
subscription_id = 'f8c5aac3-29fc-4387-858a-1f61722fb57a'
resource_group = 'forskerpl-n0ybkr-rg'
workspace_name = 'forskerpl-n0ybkr-mlw'
workspace = Workspace(subscription_id, resource_group, workspace_name)

# گرفتن datastore
datastore = Datastore.get(workspace, "researcher_data")

# گرفتن لیست همه فایل‌های Parquet
paths = [(datastore, f'Zahra/Data-07-2025/MD/MEDS_811/data/train/{i}.parquet') for i in range(36)]

total_null_dob = 0

# پردازش هر فایل به صورت جداگانه
for path in paths:
    dataset = Dataset.Tabular.from_parquet_files(path=path)
    df = dataset.to_pandas_dataframe()
    nulls = df["DOB"].isnull().sum()
    total_null_dob += nulls
    print(f"{path[1]} -> null DOBs: {nulls}")

print(f"\n✅ مجموع رکوردهای بدون DOB در همه فایل‌ها: {total_null_dob}")


Resolving access token for scope "https://storage.azure.com/.default" using identity of type "MANAGED".
Getting data access token with Assigned Identity (client_id=clientid) and endpoint type based on configuration
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}


KeyError: 'DOB'

In [8]:
from azureml.core import Workspace, Dataset, Datastore
import pandas as pd

# اتصال به AzureML workspace
subscription_id = 'f8c5aac3-29fc-4387-858a-1f61722fb57a'
resource_group = 'forskerpl-n0ybkr-rg'
workspace_name = 'forskerpl-n0ybkr-mlw'
workspace = Workspace(subscription_id, resource_group, workspace_name)

# گرفتن datastore
datastore = Datastore.get(workspace, "researcher_data")

# فقط یک فایل انتخاب کن
path = (datastore, 'Zahra/Data-07-2025/MD/MEDS_811/data/train/15.parquet')

# ساخت dataset
dataset = Dataset.Tabular.from_parquet_files(path=path)

# تبدیل به DataFrame
df = dataset.to_pandas_dataframe()

# بررسی ستون‌ها و چند سطر اول
print("🔍 ستون‌ها:", df.columns.tolist())
print("\n📊 ۵ ردیف اول:")
print(df.head(20))

# اگر ستون code وجود داره، unique ها رو بررسی کن
if "code" in df.columns:
    print("\n🧾 مقادیر یکتای ستون code:")
    print(df["code"].unique())


{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}
🔍 ستون‌ها: ['subject_id', 'time', 'code', 'numeric_value']

📊 ۵ ردیف اول:
    subject_id                time            code  numeric_value
0           36                 NaT  GENDER//Kvinde            NaN
1           36 2001-05-30 00:00:00             DOB            NaN
2           36 2017-04-18 18:00:00       M/R06AD02            NaN
3           36 2017-04-19 08:00:00       M/R06AD02            NaN
4           36 2017-04-19 18:00:00       M/R06AD02            NaN
5           36 2017-04-20 08:00:00       M/R06AD02            NaN
6           36 2017-04-20 18:00:00       M/R06AD02            NaN
7           36 2017-04-21 08:00:00       M/R06AD02            NaN
8           36 2017-04-21 18:00:00       M/R06AD02            NaN
9           36 2017-04-22 08:00:00       M/R06AD02            NaN
10          36 2017-04-22 18:00:0

In [9]:
df = dataset.to_pandas_dataframe()

# همه subject_id‌هایی که توی داده هستن
all_subjects = df["subject_id"].unique()

# کسانی که ردیف DOB دارن
dob_subjects = df[df["code"] == "DOB"]["subject_id"].unique()

# پیدا کردن بیمارانی که DOB ندارن
missing_dob = set(all_subjects) - set(dob_subjects)

print(f"🧍‍♂️ تعداد بیماران بدون رکورد DOB: {len(missing_dob)}")
if len(missing_dob) > 0:
    print("🔍 مثال‌هایی از subject_idهایی که DOB ندارن:", list(missing_dob)[:10])


{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}
🧍‍♂️ تعداد بیماران بدون رکورد DOB: 1
🔍 مثال‌هایی از subject_idهایی که DOB ندارن: [178034]


In [10]:
dob_df = df[df["code"] == "DOB"]

# پیدا کردن تعداد رکوردهایی که time خالی دارن
missing_time = dob_df["time"].isnull().sum()
print(f"📛 تعداد رکوردهای DOB که time == NaT هست: {missing_time}")

# کدوم subject_idها این مشکل رو دارن؟
null_subjects = dob_df[dob_df["time"].isnull()]["subject_id"].unique()
print(f"🧍‍♂️ تعداد بیماران با DOB ولی بدون زمان تولد: {len(null_subjects)}")
print("🔍 مثال:", list(null_subjects)[:10])


📛 تعداد رکوردهای DOB که time == NaT هست: 0
🧍‍♂️ تعداد بیماران با DOB ولی بدون زمان تولد: 0
🔍 مثال: []


In [4]:
dob_code = "DOB"  # یا هر چی که از print بالا فهمیدی

for path in paths:
    dataset = Dataset.Tabular.from_parquet_files(path=path)
    df = dataset.to_pandas_dataframe()
    
    if "code" in df.columns and "numeric_value" in df.columns:
        dob_df = df[df["code"] == dob_code]
        nulls = dob_df["numeric_value"].isnull().sum()
        total_null_dob += nulls
        print(f"{path[1]} -> null DOBs: {nulls} (total DOB records: {len(dob_df)})")
    else:
        print(f"{path[1]} ❌ ستون‌های مورد نیاز وجود ندارند.")

{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}
Zahra/Data-07-2025/MD/MEDS_811/data/train/0.parquet -> null DOBs: 49290 (total DOB records: 49290)
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}
Zahra/Data-07-2025/MD/MEDS_811/data/train/1.parquet -> null DOBs: 49290 (total DOB records: 49290)
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}
Zahra/Data-07-2025/MD/MEDS_811/data/train/2.parquet -> null DOBs: 49290 (total DOB records: 49290)
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}
Zahra/Data-07-2025/MD/MEDS_811/data

In [11]:
# پیدا کردن همه patientهایی که DOB دارن
has_dob = df[df["code"] == "DOB"]["subject_id"].unique()

# همه patientها
all_patients = df["subject_id"].unique()

# اختلاف: کسانی که DOB ندارن
missing_dob = set(all_patients) - set(has_dob)

print("🔍 بیماران بدون DOB:", missing_dob)


🔍 بیماران بدون DOB: {178034}


In [12]:
df[df["subject_id"].isin(missing_dob)]


,subject_id,time,code,numeric_value
741590,178034,NaT,GENDER//Mand,NaN
741591,178034,2022-06-24 22:49:00,ADMISSION_ADT,NaN
741592,178034,2022-06-24 22:49:00,DISCHARGE_ADT,NaN
741593,178034,2022-06-24 22:49:00,MOVE_ADT,NaN
741594,178034,2022-06-24 23:24:00,ADMISSION_ADT,NaN
741595,178034,2022-06-24 23:24:00,DISCHARGE_ADT,NaN
741596,178034,2022-06-24 23:24:00,MOVE_ADT,NaN
741597,178034,2022-06-25 00:00:00,D/DT140B,NaN
741598,178034,2022-06-25 01:10:00,ADMISSION_ADT,NaN
741599,178034,2022-06-25 01:10:00,DISCHARGE_ADT,NaN
